# 01. How `list` works

**Tier:** Deep. **Why you'd use this:** A growing buffer of results should stay fast at the end and not hide a quadratic insert at the front. The list's over-allocation and shift costs explain those timings.

**After this lesson you can:** explain why `append` stays cheap; explain the cost of `insert(0)` and `in`; read a size that jumps in steps.
About 30 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Mutability and aliasing](../02-mutability/01-aliasing-lesson.ipynb)


## Quick review

1. Why can a callee's `append` change the caller's list?
2. What does a shallow copy share with the original?

<details><summary>Answers</summary>

Both names can point at the same mutable list. Nested objects — the outer list is new, the inner ones often are not.
</details>


## The idea

A list is a dynamic array of pointers. The pointers sit in one contiguous block. The length is how many of those slots are in use. The capacity is how many slots the block has. Capacity is often larger than length, so the next `append` can write a pointer into a spare slot without asking for a new block.

When the spare slots run out, the list allocates a bigger block, copies the pointers across, and frees the old block. That copy is the expensive step. It happens rarely enough that a long run of appends is cheap per item. `insert` at the front and `in` do not get that deal: both walk the live slots every time.

<details><summary>Explain more</summary>

The object you call a list is a small header plus that block. `sys.getsizeof` reports the header and the block, not the objects the pointers reach. A list of a million integers is a list of a million pointers. The integers live elsewhere.

The byte counts below depend on the build. A 64-bit CPython and a 32-bit one will not print the same sizes. Read the shape of the steps, not a magic number.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This is `ArrayList` or a vector: geometric growth, O(1) append amortized, O(n) insert at the front. `in` is a scan, not a hash lookup.
</details>


## See it work

A sampler records readings in the order they arrive. Prepending each reading looks tidy if you want the newest first, and it gets slower as the buffer grows. Append, then reverse once, keeps arrival order reversed with one shift of the whole list at the end instead of a shift on every sample.


In [1]:
def newest_first(readings: list[int]) -> list[int]:
    # 1. Record in arrival order. Append does not shift the prefix.
    ordered = []
    for reading in readings:
        ordered.append(reading)
    # 2. One reversal puts the newest at index 0.
    ordered.reverse()
    return ordered


print(newest_first([10, 20, 30]))
print(newest_first([]))


[30, 20, 10]
[]


| call | appends build | after reverse |
| --- | --- | --- |
| newest_first([10, 20, 30]) | [10, 20, 30] | [30, 20, 10] |

```
slots: [10][20][30][ ]
         length 3, capacity at least 3
reverse swaps pointers, no new readings
```

`insert(0, reading)` on every sample would have shifted 1 pointer, then 2, then 3. The total shifts grow like the square of the number of samples. Append writes one pointer per sample. Reverse swaps each pointer once.


## Check yourself

Answer these from the example above before you run the cell.


1. Why can `getsizeof` stay the same after an append?
2. Why is a single `data[i]` cheap while `insert(0, x)` is not?
3. What does `getsizeof` leave out of a list of lists?

<details><summary>Answers</summary>

1. The block still has a free slot. The append writes a pointer there and bumps the length. No new block is allocated, so the reported size does not change.
2. Indexing is arithmetic from the start of the block. Inserting at 0 moves every pointer that is already stored. The arithmetic is one addition. The move is one copy per live element.
3. The inner lists, and everything they point at. `getsizeof` is the header and the pointer block of the outer list only.
</details>


In [2]:
data = []
for number in (1, 2, 3):
    data.insert(0, number)
print(data)


[3, 2, 1]


<details><summary>Check your answers</summary>

- The list is `[3, 2, 1]`, not `[1, 2, 3]`. Each `insert(0, number)` shifts what is already there one slot to the right and writes the new pointer at the front. The last number inserted is the one you see first.
</details>


## Why it works this way

The block is contiguous so index `i` is pointer arithmetic: base plus `i` times the size of a pointer. That is why `data[i]` is cheap. The price of contiguity is that a hole at the front does not exist. To put a new pointer at index 0, every current pointer moves one slot right. `insert(0, x)` on a list of length n moves n pointers. Do it once per new item, for n items, and you move about n²/2 pointers. `pop(0)` is the same shift the other way. `append` writes at the end, where spare capacity is, and moves nothing.

Spare capacity is why the size does not tick up on every append. The allocator asks for more slots than needed right now, in a geometric pattern. When growth happens, pointers copy into the new block once. Across n appends each pointer is copied only a small number of times, so the cost per append settles to a constant — the amortized cost. A single append that triggers growth is more expensive than its neighbors; the average across a long run is what "append is cheap" means.

`in` walks the live pointers and asks each object `==` — O(n) worst case. A set answers the same question without the scan. The list keeps order and allows duplicates; the set does not. The cost is the reason to switch.

`sys.getsizeof` reports the header plus the pointer block, not the integers or nested lists those pointers reach. CPython detail: exact byte counts and growth lengths depend on the build (volatile cell). What does not depend on the build is the shape: several appends leave the size unchanged, then it jumps, and `insert` at the front loses to `append` as the list gets longer.

The design reason for over-allocation is that asking the allocator and copying pointers is much more expensive than leaving a few unused slots. A list that grew to a million and then deleted almost everything may still hold a large block; `list(data)` or a slice copies into a tight block if you need the memory back. Most programs should not bother.


In [3]:
import sys

data = []
previous = None
jumps = []
for i in range(65):
    size = sys.getsizeof(data)
    if size != previous:
        jumps.append((len(data), size))
        previous = size
    data.append(i)
print(len(jumps))
print(jumps)


9
[(0, 56), (1, 88), (5, 120), (9, 184), (17, 248), (25, 312), (33, 376), (41, 472), (53, 568)]


In [4]:
import timeit


def by_append() -> None:
    data = []
    for i in range(4000):
        data.append(i)


def by_insert() -> None:
    data = []
    for i in range(4000):
        data.insert(0, i)


def by_contains() -> None:
    data = list(range(4000))
    found = 3999 in data
    if not found:
        raise AssertionError(found)


print(round(timeit.timeit(by_append, number=20), 4))
print(round(timeit.timeit(by_insert, number=20), 4))
print(round(timeit.timeit(by_contains, number=200), 4))


0.0017
0.0328
0.0167


## When to use it

Use `append` to build a list in the order you receive items. Reverse once if you need the opposite order. Use `insert` at the front only when the list stays tiny or the call is rare. A queue that grows at one end and shrinks at the other wants a `deque`.

Use `in` on a list when the list is short or you are already walking it. When the same list is searched over and over, build a set — you pay hash-table memory and lose order.

Do not read `sys.getsizeof` as the memory of the objects inside the list, and do not depend on a particular capacity. The spare slots are an optimization.


## Common mistake

Inserting each new reading at index 0 both reverses the samples and shifts the whole list every time. For three readings the order is already backwards. Nothing raises. The result looks like a list, so a test that only checks the length still passes.


In [5]:
def newest_wrong(readings: list[int]) -> list[int]:
    ordered = []
    for reading in readings:
        ordered.insert(0, reading)
    return ordered


print(newest_wrong([10, 20, 30]))


[30, 20, 10]


The print is `[30, 20, 10]` only if that reversed order was the goal. If the goal was arrival order, it is wrong, and the cost is the larger bug: each insert shifted a longer prefix. The fix appends, which keeps `[10, 20, 30]`. Reverse only when newest-first is actually what you want.


In [6]:
def arrival_order(readings: list[int]) -> list[int]:
    ordered = []
    for reading in readings:
        ordered.append(reading)
    return ordered


print(arrival_order([10, 20, 30]))


[10, 20, 30]


## Try it

Change `newest_first` so it still returns newest-first, but builds that order with `insert(0, reading)` instead of append-and-reverse. For a reading list of length n, how many pointer moves does the insert version perform that the append version does not, and which timing cell already shows that gap?


## What this explains

- [Lists](../../course/04-collections/01-lists-lesson.ipynb): `append` is the operation the lesson uses to grow a list. The spare capacity is why a loop of appends stays reasonable, and why `insert` at the front does not.
- [Mutability and aliasing](../02-mutability/01-aliasing-lesson.ipynb): the slots in the block are pointers. Copying a list copies those pointers. The objects they reach are not inside the block `getsizeof` measures.


## Recap

- A list is a pointer block with spare slots. `append` fills a slot. Growth copies the block occasionally, so a long run of appends is cheap per item.
- `insert(0, x)` and `pop(0)` shift every live pointer. `in` scans them.
- `getsizeof` is the block, not the objects the block points at, and the exact bytes depend on the build.

Next: [How dict and set work](../04-dicts/01-hash-table-lesson.ipynb).
